# TPC-AgentCF Colab Quickstart

This notebook avoids DeepSeek/OpenAI calls. It downloads and uses a small free local model from Hugging Face: `google/flan-t5-small`.

It now prepares two configs:
- `primary`: the paper-style default threshold (`conflict_threshold = 0.35`)
- `sensitivity`: a clearly labeled calibration run (`conflict_threshold = 0.15`) used only if the primary run produces no conflict users


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("TPC_AGENTCF_REPO_URL", "https://github.com/elom354/tpc-agentcf.git")
REPO_DIR = Path("/content/TPC-AgentCF")

if REPO_DIR.exists():
    print("Repo already present at", REPO_DIR)
else:
    !git clone "$REPO_URL" "$REPO_DIR"

%cd /content/TPC-AgentCF
print("Repo ready at", REPO_DIR)

In [ ]:
%cd /content/TPC-AgentCF
!python -m pip install --upgrade pip
!pip install "numpy<2" pytest transformers sentencepiece
!pip install -r requirements.txt

In [ ]:
import copy
import os
import yaml
from pathlib import Path

# Hard-disable remote paid endpoints for the notebook run.
for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL']:
    os.environ.pop(key, None)

default_config_path = Path('config/default.yaml')
primary_config_path = Path('config/colab_localhf_primary.yaml')
sensitivity_config_path = Path('config/colab_localhf_sensitivity.yaml')

with default_config_path.open('r', encoding='utf-8') as handle:
    base_config = yaml.safe_load(handle)

def build_config(conflict_threshold: float, escalation_threshold: float) -> dict:
    config = copy.deepcopy(base_config)
    config['llm']['backend'] = 'local_hf'
    config['llm']['model'] = 'google/flan-t5-small'
    config['llm']['max_tokens'] = 64
    config['data']['dataset'] = 'movielens'
    config['data']['max_users'] = 100
    config['data']['max_items'] = 500
    config['evaluation']['run_ablations'] = True
    config['conflict']['conflict_threshold'] = conflict_threshold
    config['conflict']['escalation_threshold'] = escalation_threshold
    return config

primary_config = build_config(conflict_threshold=0.35, escalation_threshold=0.50)
sensitivity_config = build_config(conflict_threshold=0.15, escalation_threshold=0.20)

with primary_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(primary_config, handle, sort_keys=False)
with sensitivity_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(sensitivity_config, handle, sort_keys=False)

print('Wrote', primary_config_path)
print('Wrote', sensitivity_config_path)
print('Primary threshold:', primary_config['conflict']['conflict_threshold'])
print('Sensitivity threshold:', sensitivity_config['conflict']['conflict_threshold'])


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print('Downloaded local model:', model_name)
print('Number of parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
!python -m pytest -q

In [ ]:
!python scripts/prepare_data.py --config config/colab_localhf_primary.yaml

In [ ]:
!python scripts/run_baselines.py --config config/colab_localhf_primary.yaml

In [ ]:
!python scripts/run_tpc_agentcf.py --config config/colab_localhf_primary.yaml

In [ ]:
!python scripts/run_ablation.py --config config/colab_localhf_primary.yaml

In [ ]:
!python scripts/make_paper_tables.py --config config/colab_localhf_primary.yaml

In [ ]:
import json
import pandas as pd
from pathlib import Path

recs = pd.read_json('outputs/explanations/recommendations.jsonl', lines=True)
scores = recs['conflict_score']
print('Primary run conflict score summary:')
print(scores.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print('Users above 0.35:', int((scores >= 0.35).sum()))
print('Users above 0.15:', int((scores >= 0.15).sum()))
print('This does not change the primary result. It only tells you whether a sensitivity calibration is worth running.')


## Optional Sensitivity Run

Run the next cells only if the primary threshold produces no conflict users. This is a calibration check, not the main reported result. Keep the primary and sensitivity outputs separate when interpreting results.


In [ ]:
!python scripts/run_tpc_agentcf.py --config config/colab_localhf_sensitivity.yaml
!python scripts/run_ablation.py --config config/colab_localhf_sensitivity.yaml
!python scripts/make_paper_tables.py --config config/colab_localhf_sensitivity.yaml

In [ ]:
import pandas as pd
all_results = pd.read_csv('outputs/metrics/all_users/tpc_agentcf_results.csv')
conflict_results = pd.read_csv('outputs/metrics/conflict_users/tpc_agentcf_results.csv')
high_conflict_results = pd.read_csv('outputs/metrics/high_conflict_users/tpc_agentcf_results.csv')
display(all_results)
display(conflict_results)
display(high_conflict_results)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

recs = pd.read_json('outputs/explanations/recommendations.jsonl', lines=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
recs['conflict_score'].hist(ax=axes[0], bins=20)
axes[0].set_title('Conflict Score Distribution')
axes[0].set_xlabel('conflict_score')

summary = pd.read_csv('outputs/ablations/ablation_results.csv')
pivot = summary[summary['group'] == 'all_users'][['variant', 'MRR@10']].set_index('variant')
pivot.plot(kind='bar', ax=axes[1], legend=False)
axes[1].set_title('MRR@10 by Ablation Variant')
axes[1].set_ylabel('MRR@10')
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

for path in [
    'outputs/paper_assets/research_claims.md',
    'outputs/paper_assets/main_table.md',
    'outputs/paper_assets/qualitative_examples.md',
]:
    print('\n' + '=' * 80)
    print(path)
    print('=' * 80)
    print(Path(path).read_text(encoding='utf-8'))